# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# --- Install + auth (same as before) ---
!pip install huggingface_hub pandas pyarrow scikit-learn -q
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import pandas as pd
import numpy as np

# --- Load March 2026 + dimension tables ---
df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)
dim_content = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_content.parquet")

# --- Rebuild working df (same filters as ML-05/06/07) ---
df = df_march.copy()
df = df.merge(
    dim_content[['content_hash_id', 'content_type', 'word_count', 'content_created_date']],
    on='content_hash_id', how='left'
)
df = df[df['gsc_data_available'] == True]
df = df[df['gsc_impressions'] >= 500]

print(df.shape)

(101451, 34)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



**Question shape:** My lane's label, `is_opportunity`, is a yes/no observed label built from
within-client percentile ranks of impressions and CTR (from ML-05). Per the training-honest-models
skill's method table, a yes/no label with an observed target maps to: "Logistic Regression, then
Random Forest."

**Why start with Logistic Regression:** It's the simplest model that fits this question shape,
and per the skill, "simplicity is a feature" — a readable model that's 2 points weaker than an
opaque one still teaches more. Logistic Regression also gives clean, interpretable coefficients,
which lets me sanity-check whether the model is leaning on sensible signals (like position) or
something suspicious.

**Why add Random Forest second:** Random Forest can capture non-linear relationships and feature
interactions that Logistic Regression can't (e.g., maybe poor position only matters when combined
with high impressions — exactly my baseline rule's logic). Comparing both against the same
baseline shows whether added model complexity actually earns its keep, per the skill's warning
not to "reward complexity alone."

**Why not Gradient Boosting or clustering:** Gradient Boosting is listed as "where safe" —
given my modest honest-feature signal (ML-05's grouped CV AUC was 0.637, not near-perfect),
a heavier model risks overfitting on a signal this size without much practical upside.
Clustering doesn't fit — my question is binary opportunity scoring, not archetype discovery,
which is a different lane.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Section 2: Split Design

**Split type: Grouped by client_hash_id (GroupKFold), not random, not time-based.**

**Why grouped, not random:** Rows from the same client share hidden character — pricing tier,
industry, content strategy, baseline traffic level. A random split would let rows from the same
client appear in both train and test, letting the model partly memorize client-specific quirks
rather than learn a generalizable opportunity signal. The honest question is "does this work on
a client the model has never seen?" — only a grouped split answers that.

**Why not time-based:** My data is a single month (March 2026). There's no meaningful
train-on-past/test-on-future split within one month without arbitrarily slicing days, which would
just be a weaker, noisier version of the same client-generalization question.

**Consistency with prior work:** This is the same GroupKFold-by-client_hash_id design used in
ML-05's leakage self-test, so what "honest" means stays consistent across the whole project.

**Fold count:** 5-fold GroupKFold, matching ML-05.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np

RANDOM_SEED = 42

# ============================================================
# 1. Rebuild label (same as ML-05)
# ============================================================
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions']
df['impr_rank'] = df.groupby('client_hash_id')['gsc_impressions'].rank(pct=True)
df['engagement_rank'] = df.groupby('client_hash_id')['ctr'].rank(pct=True)
df['is_opportunity'] = ((df['impr_rank'] >= 0.67) & (df['engagement_rank'] <= 0.33)).astype(int)

# ============================================================
# 2. Rebuild honest features (same as ML-05)
# ============================================================
df = pd.get_dummies(df, columns=['content_type'], prefix='ctype')

df['has_word_count'] = df['word_count'].notna().astype(int)
df['word_count_filled'] = df['word_count'].fillna(0)

df['content_age_days'] = (
    pd.to_datetime(df['report_date']) - pd.to_datetime(df['content_created_date'])
).dt.days

df['has_position_data'] = (df['gsc_avg_position'] != 0).astype(int)
df['gsc_avg_position_filled'] = df['gsc_avg_position'].replace(0, np.nan).fillna(df['gsc_avg_position'].median())

df['has_ga4_access'] = (df['ga4_data_available'] == True).astype(int)
df['pct_sessions_organic'] = (df['sessions_organic'] / df['ga4_sessions'].replace(0, np.nan)).fillna(0)
df['pct_sessions_ai'] = (df['sessions_ai'] / df['ga4_sessions'].replace(0, np.nan)).fillna(0)

feature_cols = [
    'has_word_count', 'word_count_filled', 'content_age_days',
    'has_position_data', 'gsc_avg_position_filled',
    'has_ga4_access', 'pct_sessions_organic', 'pct_sessions_ai',
] + [c for c in df.columns if c.startswith('ctype_')]

X = df[feature_cols].copy()
y = df['is_opportunity']
groups = df['client_hash_id']

numeric_cols = ['word_count_filled', 'content_age_days', 'gsc_avg_position_filled',
                 'pct_sessions_organic', 'pct_sessions_ai']

# ============================================================
# 3. Rebuild ML-07 baseline SCORE (global ranking, as originally submitted)
# ============================================================
df['baseline_impr_pct'] = df['gsc_impressions'].rank(pct=True)
df['baseline_position_badness_pct'] = df['gsc_avg_position'].rank(pct=True)
df['baseline_score'] = df['baseline_impr_pct'] * df['baseline_position_badness_pct']

# ============================================================
# 4. Base rate
# ============================================================
print(f"Base rate (positive class): {y.mean():.3f}")

# ============================================================
# 5. GroupKFold: evaluate baseline, Logistic Regression, Random Forest — same split, same folds
# ============================================================
gkf = GroupKFold(n_splits=5)

baseline_aucs, logreg_aucs, rf_aucs = [], [], []
oof_baseline, oof_logreg, oof_rf, oof_y = [], [], [], []

for train_idx, test_idx in gkf.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    baseline_test_score = df['baseline_score'].iloc[test_idx]
    baseline_aucs.append(roc_auc_score(y_test, baseline_test_score))

    logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    logreg.fit(X_train, y_train)
    logreg_pred = logreg.predict_proba(X_test)[:, 1]
    logreg_aucs.append(roc_auc_score(y_test, logreg_pred))

    rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict_proba(X_test)[:, 1]
    rf_aucs.append(roc_auc_score(y_test, rf_pred))

    oof_baseline.extend(baseline_test_score.tolist())
    oof_logreg.extend(logreg_pred.tolist())
    oof_rf.extend(rf_pred.tolist())
    oof_y.extend(y_test.tolist())

# ============================================================
# 6. Comparison table
# ============================================================
comparison = pd.DataFrame({
    'Method': ['Baseline (ML-07 rule)', 'Logistic Regression', 'Random Forest'],
    'Mean AUC': [np.mean(baseline_aucs), np.mean(logreg_aucs), np.mean(rf_aucs)],
    'Std AUC': [np.std(baseline_aucs), np.std(logreg_aucs), np.std(rf_aucs)],
})
print("\n=== Comparison Table (5-fold GroupKFold by client_hash_id) ===")
print(comparison.to_string(index=False))
print(f"\nBase rate: {y.mean():.3f}")

# ============================================================
# 7. Precision@50 on pooled out-of-fold predictions
# ============================================================
oof_df = pd.DataFrame({'y': oof_y, 'baseline': oof_baseline, 'logreg': oof_logreg, 'rf': oof_rf})

def precision_at_k(df_scores, score_col, k=50):
    top_k = df_scores.sort_values(score_col, ascending=False).head(k)
    return top_k['y'].mean()

for col in ['baseline', 'logreg', 'rf']:
    p50 = precision_at_k(oof_df, col, k=50)
    print(f"Precision@50 ({col}): {p50:.3f}")


Base rate (positive class): 0.111

=== Comparison Table (5-fold GroupKFold by client_hash_id) ===
               Method  Mean AUC  Std AUC
Baseline (ML-07 rule)  0.806221 0.048528
  Logistic Regression  0.636789 0.033734
        Random Forest  0.655781 0.033265

Base rate: 0.111
Precision@50 (baseline): 0.500
Precision@50 (logreg): 0.300
Precision@50 (rf): 0.200


## Section 3: Train + Compare vs Baseline

**Comparison table (5-fold GroupKFold by client_hash_id, same split used for all three methods):**

| Method | Mean AUC | Std AUC | Precision@50 |
|---|---|---|---|
| Baseline (ML-07 rule) | 0.806 | 0.049 | 0.500 |
| Logistic Regression | 0.637 | 0.034 | 0.300 |
| Random Forest | 0.656 | 0.033 | 0.200 |

**Base rate:** 0.111 (11.1% of rows are labeled is_opportunity = 1).

**Reading the table:** The baseline rule outperforms both trained models on every metric. This
looks surprising at first, but it has a specific, explainable cause tied directly to how the
label was constructed, not a flaw in the modeling process.

**Random Forest vs Logistic Regression:** Random Forest (0.656) does modestly outperform
Logistic Regression (0.637) on AUC, consistent with the expectation that it can capture feature
interactions a linear model can't. Neither model comes close to the baseline, for the reason
explained below.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Section 4: Errors and Interpretation

**Why the baseline wins — a structural asymmetry, not a modeling failure:**

The is_opportunity label is built directly from gsc_impressions (the impression-rank half of
the label formula, from ML-05). My baseline rule (ML-07) also uses gsc_impressions directly as
one of its two scoring ingredients. This means the baseline has direct access to a raw signal
that is mechanically close to how the label itself was constructed — the two are correlated
almost by definition.

My trained models (Logistic Regression, Random Forest), by contrast, were built following the
leakage rules established in ML-05: gsc_impressions is a banned feature, since it's a label
ingredient. The models were deliberately restricted to leakage-safe features (position,
word count, content age, traffic mix, content type) — none of which have as direct a
relationship to the label as raw impressions does.

This is a genuine finding worth stating plainly: the baseline's apparent strength comes from
using label-adjacent information the honest model deliberately avoids, not from being a smarter
rule. It is not an apples-to-apples comparison of "simple rule vs. smart model" — it's a
comparison of "a rule using a near-label-strength signal" vs. "a model restricted to
leakage-safe signals only."

**What the models lean on:** Both models had to work only with the 11 leakage-safe features,
where the strongest confirmed real-world signal (from ML-06) was ranking position — a much
weaker predictor of this specific label than raw impressions is.

**Interpretation for the capstone:** this is a case where "beating the baseline" isn't the
right takeaway — the baseline's advantage is an artifact of what it's allowed to use, not
evidence that simple rules are better than learned models. Documenting this honestly is more
valuable than hiding it or re-engineering the baseline to close the gap artificially.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.